# Source Data Profiling

This notebook profiles the simulated EHR, HR, and Credentialing provider datasets before any cleaning or standardization is applied.

The goal is to document source-level data quality issues, identify formatting differences that will need standardization, and flag anything that may affect matching later in the MDM workflow. Raw source values are not changed here.

## 1. Setup

In [1]:
import pandas as pd
from pathlib import Path

SIMULATED_DIR = Path("../data/simulated")

## 2. Load the source data

In [2]:
ehr_raw = pd.read_csv(SIMULATED_DIR / "ehr_providers.csv")
hr_raw = pd.read_csv(SIMULATED_DIR / "hr_providers.csv")
cred_raw = pd.read_csv(SIMULATED_DIR / "credentialing_providers.csv")

## 3. Set up the findings log

Each finding is assigned to the stage where it will be handled:

- **Cleaning** — safe datatype or structural corrections
- **Standardization** — consistent representation of the same type of value
- **Data Quality** — completeness or source-quality issues that should be retained and documented
- **Matching** — observations that affect how strongly a field can be used for record matching

In [3]:
findings = pd.DataFrame(columns=[
    "dataset",
    "column",
    "issue",
    "example",
    "planned_action",
    "stage",
    "status"
])

# Add a finding only if the same dataset / column / issue is not already logged.
def add_finding(dataset, column, issue, example, planned_action, stage):
    duplicate = (
        (findings["dataset"] == dataset) &
        (findings["column"] == column) &
        (findings["issue"] == issue)
    ).any()

    if duplicate:
        print("Finding already exists.")
        return

    findings.loc[len(findings)] = {
        "dataset": dataset,
        "column": column,
        "issue": issue,
        "example": example,
        "planned_action": planned_action,
        "stage": stage,
        "status": "Open"
    }


# Remove a finding if it was logged by mistake or is no longer needed.
def delete_finding(dataset, column, issue):
    global findings

    findings = findings[
        ~(
            (findings["dataset"] == dataset) &
            (findings["column"] == column) &
            (findings["issue"] == issue)
        )
    ].reset_index(drop=True)

## 4. Inspect the datasets

In [4]:
simulated_providers = {
    "EHR": ehr_raw,
    "HR": hr_raw,
    "CREDENTIALING": cred_raw
}

# Confirm row counts and source schemas before profiling individual fields.
for name, df in simulated_providers.items():
    print(name)
    print(df.shape)
    print(df.columns)
    print()

EHR
(3000, 13)
Index(['ehr_provider_id', 'npi', 'first_name', 'middle_name', 'last_name',
       'credential', 'specialty_code', 'address_line_1', 'address_line_2',
       'city', 'state', 'zip', 'phone'],
      dtype='str')

HR
(2250, 10)
Index(['employee_id', 'npi', 'first_name', 'middle_initial', 'last_name',
       'job_credential', 'job_specialty_code', 'work_city', 'work_state',
       'work_phone'],
      dtype='str')

CREDENTIALING
(2550, 15)
Index(['credentialing_id', 'npi', 'legal_first_name', 'legal_middle_name',
       'legal_last_name', 'credential', 'taxonomy_code', 'license_number',
       'license_state', 'address_line_1', 'address_line_2', 'city', 'state',
       'zip', 'phone'],
      dtype='str')



## 5. Validate inferred datatypes

In [5]:
for name, df in simulated_providers.items():
    print(name)
    print(df.dtypes)
    print()

# NPI is an identifier, but pandas inferred it as numeric because of the source values.
for dataset in ["EHR", "HR", "Credentialing"]:
    add_finding(
        dataset=dataset,
        column="npi",
        issue="NPI inferred as numeric during import",
        example="1234567890.0",
        planned_action="Convert to string and validate 10-digit format",
        stage="Cleaning"
    )

# ZIP is also an identifier and should not be treated as a numeric measure.
for dataset in ["EHR", "Credentialing"]:
    add_finding(
        dataset=dataset,
        column="zip",
        issue="ZIP inferred as numeric during import",
        example="773386118",
        planned_action="Convert to string before ZIP standardization",
        stage="Cleaning"
    )

# Credentialing phone values were inferred as numeric.
add_finding(
    dataset="Credentialing",
    column="phone",
    issue="Phone inferred as numeric during import",
    example="2.816416e+09",
    planned_action="Convert to string and validate phone length",
    stage="Cleaning"
)

EHR
ehr_provider_id        str
npi                float64
first_name             str
middle_name            str
last_name              str
credential             str
specialty_code         str
address_line_1         str
address_line_2         str
city                   str
state                  str
zip                  int64
phone                  str
dtype: object

HR
employee_id               str
npi                   float64
first_name                str
middle_initial            str
last_name                 str
job_credential            str
job_specialty_code        str
work_city                 str
work_state                str
work_phone                str
dtype: object

CREDENTIALING
credentialing_id         str
npi                  float64
legal_first_name         str
legal_middle_name        str
legal_last_name          str
credential               str
taxonomy_code            str
license_number           str
license_state            str
address_line_1           str
address_

## 6. Check missingness

In [6]:
# Review all columns with missing values in each source.
for name, df in simulated_providers.items():
    print(f"\n{name} Missingness")

    missing_summary = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2)
    })

    display(
        missing_summary[
            missing_summary["missing_count"] > 0
        ].sort_values("missing_pct", ascending=False)
    )


EHR Missingness


,missing_count,missing_pct
address_line_2,2667,88.90
middle_name,1468,48.93
credential,836,27.87
npi,238,7.93
phone,156,5.20



HR Missingness


,missing_count,missing_pct
npi,1206,53.60
middle_initial,1161,51.60
job_credential,620,27.56
job_specialty_code,481,21.38
work_phone,279,12.40



CREDENTIALING Missingness


,missing_count,missing_pct
address_line_2,2201,86.31
legal_middle_name,1170,45.88
credential,708,27.76
license_number,421,16.51
license_state,371,14.55
phone,191,7.49
npi,51,2.00


In [7]:
# Log missing NPI because it directly affects deterministic matching.
for dataset, df in simulated_providers.items():
    missing_count = df["npi"].isna().sum()
    missing_pct = round(df["npi"].isna().mean() * 100, 2)

    if missing_count > 0:
        add_finding(
            dataset=dataset,
            column="npi",
            issue="NPI contains missing values",
            example=f"{missing_count} records ({missing_pct}% missing)",
            planned_action="Retain missing values; do not impute NPI; use other standardized attributes for matching",
            stage="Matching"
        )

# HR specialty completeness is materially lower than the other source fields used for provider identity.
hr_specialty_missing = hr_raw["job_specialty_code"].isna().sum()
hr_specialty_missing_pct = round(hr_raw["job_specialty_code"].isna().mean() * 100, 2)

if hr_specialty_missing > 0:
    add_finding(
        dataset="HR",
        column="job_specialty_code",
        issue="Specialty code contains missing values",
        example=f"{hr_specialty_missing} records ({hr_specialty_missing_pct}% missing)",
        planned_action="Retain missing values and use specialty from other available sources during matching and survivorship",
        stage="Data Quality"
    )

Missing values are not automatically treated as defects. Fields such as middle name and address line 2 can be legitimately blank, so they are profiled above but are not logged unless the missingness affects a later MDM rule.

## 7. Check duplicates and source-key uniqueness

In [8]:
# Check for exact duplicate rows across each source.
for name, df in simulated_providers.items():
    duplicates = df[df.duplicated(keep=False)]
    print(f"{name}: {len(duplicates)} duplicated rows")
    display(duplicates.head())

EHR: 0 duplicated rows


,ehr_provider_id,npi,first_name,middle_name,last_name,credential,specialty_code,address_line_1,address_line_2,city,state,zip,phone


HR: 0 duplicated rows


,employee_id,npi,first_name,middle_initial,last_name,job_credential,job_specialty_code,work_city,work_state,work_phone


CREDENTIALING: 0 duplicated rows


,credentialing_id,npi,legal_first_name,legal_middle_name,legal_last_name,credential,taxonomy_code,license_number,license_state,address_line_1,address_line_2,city,state,zip,phone


### EHR

In [9]:
# Source record ID should be unique.
display(
    ehr_raw[
        ehr_raw["ehr_provider_id"].duplicated(keep=False)
    ].sort_values("ehr_provider_id")
)

# Ignore missing NPI values when checking whether populated NPIs repeat.
display(
    ehr_raw[
        ehr_raw["npi"].notna() &
        ehr_raw["npi"].duplicated(keep=False)
    ].sort_values("npi")
)

# Phone numbers can legitimately be shared by providers at the same practice or facility.
phone_counts_ehr = ehr_raw["phone"].value_counts()
display(phone_counts_ehr[phone_counts_ehr > 1])

add_finding(
    dataset="EHR",
    column="phone",
    issue="Phone numbers are shared across multiple provider records",
    example="(210) 617-5300 appears on 9 records",
    planned_action="Retain values and use phone as a supporting match attribute only",
    stage="Matching"
)

,ehr_provider_id,npi,first_name,middle_name,last_name,credential,specialty_code,address_line_1,address_line_2,city,state,zip,phone


,ehr_provider_id,npi,first_name,middle_name,last_name,credential,specialty_code,address_line_1,address_line_2,city,state,zip,phone


phone
(210) 617-5300    9
(855) 223-7123    9
(713) 792-6161    9
(972) 715-5000    8
(713) 620-4000    7
                 ..
(254) 732-2262    2
(210) 397-8500    2
8774182978        2
(469) 204-2021    2
9566657049        2
Name: count, Length: 122, dtype: int64

In [10]:
# ZIP codes are expected to repeat across providers and should not be treated as unique identifiers.
zip_counts_ehr = ehr_raw["zip"].value_counts()
display(zip_counts_ehr[zip_counts_ehr > 1])

add_finding(
    dataset="EHR",
    column="zip",
    issue="ZIP codes are shared across multiple provider records",
    example="782294402 appears on 15 records",
    planned_action="Retain values and use ZIP as a supporting match attribute only",
    stage="Matching"
)

zip
782294402    15
782344504    14
782294404    13
753907201    13
770301501    12
             ..
770302777     2
780068546     2
785392909     2
770302761     2
782369908     2
Name: count, Length: 226, dtype: int64

### HR

In [11]:
display(
    hr_raw[
        hr_raw["employee_id"].duplicated(keep=False)
    ].sort_values("employee_id")
)

display(
    hr_raw[
        hr_raw["npi"].notna() &
        hr_raw["npi"].duplicated(keep=False)
    ].sort_values("npi")
)

phone_counts_hr = hr_raw["work_phone"].value_counts()
display(phone_counts_hr[phone_counts_hr > 1])

add_finding(
    dataset="HR",
    column="work_phone",
    issue="Phone numbers are shared across multiple provider records",
    example="713-792-6161 appears on 9 records",
    planned_action="Retain values and use phone as a supporting match attribute only",
    stage="Matching"
)

,employee_id,npi,first_name,middle_initial,last_name,job_credential,job_specialty_code,work_city,work_state,work_phone


,employee_id,npi,first_name,middle_initial,last_name,job_credential,job_specialty_code,work_city,work_state,work_phone


work_phone
713-792-6161    9
210-617-5300    7
855-223-7123    7
817-335-3022    6
972-715-5000    6
               ..
817-702-1244    2
855-984-5121    2
7137911414      2
512-693-7045    2
888-804-3000    2
Name: count, Length: 74, dtype: int64

### Credentialing

In [12]:
display(
    cred_raw[
        cred_raw["credentialing_id"].duplicated(keep=False)
    ].sort_values("credentialing_id")
)

display(
    cred_raw[
        cred_raw["npi"].notna() &
        cred_raw["npi"].duplicated(keep=False)
    ].sort_values("npi")
)

display(
    cred_raw[
        cred_raw["license_number"].notna() &
        cred_raw["license_number"].duplicated(keep=False)
    ].sort_values("license_number")
)

zip_counts_cred = cred_raw["zip"].value_counts()
display(zip_counts_cred[zip_counts_cred > 1])

add_finding(
    dataset="Credentialing",
    column="zip",
    issue="ZIP codes are shared across multiple provider records",
    example="782344504 appears on 13 records",
    planned_action="Retain values and use ZIP as a supporting match attribute only",
    stage="Matching"
)

,credentialing_id,npi,legal_first_name,legal_middle_name,legal_last_name,credential,taxonomy_code,license_number,license_state,address_line_1,address_line_2,city,state,zip,phone


,credentialing_id,npi,legal_first_name,legal_middle_name,legal_last_name,credential,taxonomy_code,license_number,license_state,address_line_1,address_line_2,city,state,zip,phone


,credentialing_id,npi,legal_first_name,legal_middle_name,legal_last_name,credential,taxonomy_code,license_number,license_state,address_line_1,address_line_2,city,state,zip,phone


zip
782344504    13
753907201    13
770301501    12
770304000    11
782294402    11
             ..
780415955     2
760537209     2
765485725     2
760865705     2
785031241     2
Name: count, Length: 189, dtype: int64

## 8. Profile formatting and representation

These checks document differences in how equivalent provider information is represented. The raw values stay unchanged in this notebook; the findings will drive the standardization step.

### 8.1 Names

In [13]:
# Review capitalization patterns without changing the raw values.
name_columns = {
    "EHR": ["first_name", "middle_name", "last_name"],
    "HR": ["first_name", "middle_initial", "last_name"],
    "CREDENTIALING": ["legal_first_name", "legal_middle_name", "legal_last_name"]
}

for dataset, columns in name_columns.items():
    df = simulated_providers[dataset]

    print(f"\n{dataset}")
    for column in columns:
        values = df[column].dropna().astype(str).str.strip()
        print(
            column,
            {
                "upper": int(values.str.isupper().sum()),
                "title": int(values.str.istitle().sum()),
                "lower": int(values.str.islower().sum())
            }
        )


EHR
first_name {'upper': 2265, 'title': 738, 'lower': 0}
middle_name {'upper': 1532, 'title': 1081, 'lower': 0}
last_name {'upper': 2240, 'title': 760, 'lower': 0}

HR
first_name {'upper': 2250, 'title': 3, 'lower': 0}
middle_initial {'upper': 1089, 'title': 972, 'lower': 0}
last_name {'upper': 2250, 'title': 0, 'lower': 0}

CREDENTIALING
legal_first_name {'upper': 2550, 'title': 2, 'lower': 0}
legal_middle_name {'upper': 1380, 'title': 554, 'lower': 0}
legal_last_name {'upper': 2550, 'title': 0, 'lower': 0}


In [14]:
# Name capitalization is inconsistent across the simulated source systems.
for dataset, columns in name_columns.items():
    add_finding(
        dataset=dataset,
        column=", ".join(columns),
        issue="Name capitalization is inconsistent",
        example="Bray, TAYLOR",
        planned_action="Standardize name casing while retaining the raw source values",
        stage="Standardization"
    )

In [15]:
# Middle-name fields contain both initials and full names.
print("EHR middle-name lengths")
display(ehr_raw["middle_name"].dropna().astype(str).str.strip().str.len().value_counts().sort_index())

print("Credentialing legal middle-name lengths")
display(cred_raw["legal_middle_name"].dropna().astype(str).str.strip().str.len().value_counts().sort_index())

add_finding(
    dataset="EHR",
    column="middle_name",
    issue="Middle name contains both initials and full names",
    example="Single-character initials and full middle names are both present",
    planned_action="Preserve the full raw value and derive a comparable middle initial for matching",
    stage="Standardization"
)

add_finding(
    dataset="Credentialing",
    column="legal_middle_name",
    issue="Legal middle name contains both initials and full names",
    example="Single-character initials and full middle names are both present",
    planned_action="Preserve the full raw value and derive a comparable middle initial for matching",
    stage="Standardization"
)

EHR middle-name lengths


middle_name
1     1055
2       27
3       25
4       79
5      101
6       86
7       71
8       36
9       29
10       7
11       7
12       6
13       1
15       2
Name: count, dtype: int64

Credentialing legal middle-name lengths


legal_middle_name
1     508
2      49
3      77
4     144
5     173
6     163
7     135
8      70
9      32
10     12
11      9
12      5
13      1
15      1
16      1
Name: count, dtype: int64

### 8.2 Credentials

In [16]:
# Review the most common credential representations in each source.
credential_columns = {
    "EHR": "credential",
    "HR": "job_credential",
    "CREDENTIALING": "credential"
}

for dataset, column in credential_columns.items():
    print(f"\n{dataset} - {column}")
    display(
        simulated_providers[dataset][column]
        .dropna()
        .astype(str)
        .value_counts()
        .head(25)
    )


EHR - credential


credential
MD         287
M.D.       212
LPC         91
DDS         64
PA-C        55
PHARMD      43
CRNA        40
LCSW        39
RN          39
DO          34
FNP         34
FNP-C       33
NP          28
LVN         28
DC          27
D.D.S.      27
D.C.        27
OTR         26
PT          21
BCBA        20
D.O.        19
PH.D.       18
PTA         18
PT, DPT     18
DPT         16
Name: count, dtype: int64


HR - job_credential


job_credential
MD         203
M.D.       160
LPC         69
PA-C        47
DDS         46
CRNA        34
RN          31
PHARMD      30
LCSW        27
FNP         26
OTR         23
D.C.        23
DO          22
DC          19
LVN         19
NP          19
FNP-C       19
PT          17
BCBA        17
PT, DPT     16
D.D.S.      16
PTA         14
D.O.        14
APRN        13
LMSW        13
Name: count, dtype: int64


CREDENTIALING - credential


credential
MD         242
M.D.       174
LPC         74
DDS         54
PA-C        48
PHARMD      38
CRNA        37
RN          35
DO          29
FNP         29
FNP-C       29
D.D.S.      27
LCSW        27
NP          26
D.C.        23
OTR         23
DC          23
LVN         21
PT          19
D.O.        16
PT, DPT     16
PH.D.       16
BCBA        15
PTA         15
APRN        14
Name: count, dtype: int64

In [17]:
# Equivalent credentials appear with punctuation differences.
for dataset, column in credential_columns.items():
    add_finding(
        dataset=dataset,
        column=column,
        issue="Credential abbreviations use inconsistent punctuation",
        example="MD, M.D.",
        planned_action="Standardize equivalent credential abbreviations to a consistent representation",
        stage="Standardization"
    )

# Some records contain more than one credential and the separators are not always consistent.
add_finding(
    dataset="EHR",
    column="credential",
    issue="Some records contain multiple credentials",
    example="APRN, AGACNP-BC",
    planned_action="Standardize separators while preserving all credential values",
    stage="Standardization"
)

add_finding(
    dataset="Credentialing",
    column="credential",
    issue="Multiple credentials are not always separated consistently",
    example="M.S. CCC-SLP",
    planned_action="Standardize separators while preserving all credential values",
    stage="Standardization"
)

### 8.3 Addresses

In [18]:
# Review common address values before defining abbreviation rules.
for column in ["address_line_1", "address_line_2"]:
    print(f"\nEHR - {column}")
    display(ehr_raw[column].dropna().astype(str).value_counts().head(20))

    print(f"Credentialing - {column}")
    display(cred_raw[column].dropna().astype(str).value_counts().head(20))


EHR - address_line_1


address_line_1
5323 HARRY HINES BLVD       27
1515 HOLCOMBE BLVD          16
4502 MEDICAL DR             13
7400 MERTON MINTER ST       13
3551 ROGER BROOKE DR        13
301 UNIVERSITY BLVD         12
2002 HOLCOMBE BLVD          12
6621 FANNIN ST              12
1504 TAUB LOOP              10
6701 FANNIN ST              10
7703 FLOYD CURL DR           8
12222 MERIT DR STE 600       8
1500 S MAIN ST               7
6431 FANNIN ST               7
8915 HARRY HINES BLVD        7
9846 HWY 31 E                7
3840 HULEN ST                7
1935 MEDICAL DISTRICT DR     7
2401 S 31ST ST               7
4500 S LANCASTER RD          7
Name: count, dtype: int64

Credentialing - address_line_1


address_line_1
5323 HARRY HINES BLVD     21
1515 HOLCOMBE BLVD        14
3551 ROGER BROOKE DR      12
301 UNIVERSITY BLVD       11
2002 HOLCOMBE BLVD        10
6621 FANNIN ST            10
7400 MERTON MINTER ST     10
4502 MEDICAL DR           10
6701 FANNIN ST             9
1504 TAUB LOOP             9
7703 FLOYD CURL DR         8
1500 S MAIN ST             7
4500 S LANCASTER RD        7
9846 HWY 31 E              7
6431 FANNIN ST             7
2401 S 31ST ST             7
12222 MERIT DR STE 600     6
8915 HARRY HINES BLVD      6
700 MILAM ST STE 1300      5
3840 HULEN ST              5
Name: count, dtype: int64


EHR - address_line_2


address_line_2
SUITE 100    14
SUITE 200    13
SUITE 300    10
SUITE 101     6
STE. 300      4
STE 100       4
SUITE 500     4
SUITE B       3
SUITE 110     3
SUITE C       3
SUITE 107     3
SUITE 208     3
SUITE 230     2
SUITE 570     2
SUITE 400     2
SUITE 109     2
STE 200       2
SUITE 201     2
STE 102       2
SUITE 112     2
Name: count, dtype: int64

Credentialing - address_line_2


address_line_2
SUITE 100     17
SUITE 200     11
SUITE 300      8
SUITE 101      5
STE 100        5
SUITE 500      4
SUITE B        4
SUITE 110      4
SUITE C        4
SUITE 208      3
STE 102        3
SUITE 150      3
SUITE 112      3
SUITE 107      3
SUITE 130      3
SUITE 400      3
SUITE 108      3
SUITE 202      2
400            2
SUITE 3100     2
Name: count, dtype: int64

In [19]:
for dataset in ["EHR", "Credentialing"]:
    add_finding(
        dataset=dataset,
        column="address_line_1",
        issue="Street suffix abbreviations are inconsistent",
        example="26006 OAKRIDGE DR., 4502 MEDICAL DR",
        planned_action="Standardize common street suffix abbreviations and punctuation",
        stage="Standardization"
    )

    add_finding(
        dataset=dataset,
        column="address_line_2",
        issue="Suite and unit representations are inconsistent",
        example="SUITE 100, STE # 208, SUITE # 9",
        planned_action="Standardize suite and unit representations while retaining the raw value",
        stage="Standardization"
    )

### 8.4 Phone

In [20]:
# Check phone lengths after converting values to strings for inspection only.
print("EHR phone lengths")
display(ehr_raw["phone"].dropna().astype(str).str.len().value_counts())

print("HR phone lengths")
display(hr_raw["work_phone"].dropna().astype(str).str.len().value_counts())

print("Credentialing phone lengths")
display(cred_raw["phone"].dropna().astype(str).str.len().value_counts())

EHR phone lengths


phone
14    1959
10     885
Name: count, dtype: int64

HR phone lengths


work_phone
12    1587
10     384
Name: count, dtype: int64

Credentialing phone lengths


phone
12    2358
11       1
Name: count, dtype: int64

In [21]:
add_finding(
    dataset="EHR",
    column="phone",
    issue="Phone numbers exist in varied formats",
    example="(210) 617-5300, 8774182978",
    planned_action="Standardize phone numbers to a digits-only comparable format",
    stage="Standardization"
)

add_finding(
    dataset="HR",
    column="work_phone",
    issue="Phone numbers exist in varied formats",
    example="817-702-1244, 7137911414",
    planned_action="Standardize phone numbers to a digits-only comparable format",
    stage="Standardization"
)

add_finding(
    dataset="Credentialing",
    column="phone",
    issue="Phone requires normalization after conversion from numeric",
    example="2.816416e+09",
    planned_action="Convert to string, remove numeric artifacts, and standardize to a comparable phone format",
    stage="Standardization"
)

### 8.5 ZIP

In [22]:
print("EHR ZIP lengths")
display(ehr_raw["zip"].dropna().astype(str).str.len().value_counts())

print("Credentialing ZIP lengths")
display(cred_raw["zip"].dropna().astype(str).str.len().value_counts())

EHR ZIP lengths


zip
9    2795
5     205
Name: count, dtype: int64

Credentialing ZIP lengths


zip
9    2372
5     178
Name: count, dtype: int64

In [23]:
for dataset in ["EHR", "Credentialing"]:
    add_finding(
        dataset=dataset,
        column="zip",
        issue="ZIP has mixed 5-digit and 9-digit formats",
        example="77380, 774336767",
        planned_action="Preserve ZIP+4 when available and derive a 5-digit ZIP for comparison and matching",
        stage="Standardization"
    )

### 8.6 Specialty / taxonomy

In [24]:
# Compare specialty availability across equivalent source fields.
specialty_columns = {
    "EHR": "specialty_code",
    "HR": "job_specialty_code",
    "CREDENTIALING": "taxonomy_code"
}

for dataset, column in specialty_columns.items():
    print(f"\n{dataset} - {column}")
    print("Missing:", simulated_providers[dataset][column].isna().sum())
    display(simulated_providers[dataset][column].dropna().astype(str).value_counts().head(15))


EHR - specialty_code
Missing: 0


specialty_code
106S00000X    186
101YP2500X    143
363LF0000X    136
183500000X    135
235Z00000X    128
390200000X    115
225100000X     92
207Q00000X     82
363A00000X     73
1223G0001X     72
1041C0700X     68
163W00000X     63
111N00000X     62
101YM0800X     61
225X00000X     59
Name: count, dtype: int64


HR - job_specialty_code
Missing: 481


job_specialty_code
106S00000X    107
101YP2500X     86
363LF0000X     79
235Z00000X     75
183500000X     72
390200000X     58
225100000X     58
207Q00000X     51
363A00000X     49
1041C0700X     44
225X00000X     40
111N00000X     40
1223G0001X     39
163W00000X     38
101YM0800X     37
Name: count, dtype: int64


CREDENTIALING - taxonomy_code
Missing: 0


taxonomy_code
106S00000X    166
363LF0000X    118
183500000X    115
101YP2500X    112
235Z00000X    106
390200000X     99
225100000X     83
207Q00000X     73
363A00000X     65
1223G0001X     63
163W00000X     55
111N00000X     53
1041C0700X     53
101YM0800X     53
207R00000X     53
Name: count, dtype: int64

## 9. Compare equivalent fields across sources

The source systems use different column names for the same provider attributes. This section compares populated values by NPI to identify differences that may need standardization before matching.

In [27]:
def compare_across_sources(left_name, left_df, left_column, right_name, right_df, right_column):
    # Rename the compared fields before merging so column names are predictable
    # even when the source columns have different names.
    left_value_col = f"{left_column}_{left_name.lower()}"
    right_value_col = f"{right_column}_{right_name.lower()}"

    left = (
        left_df[["npi", left_column]]
        .dropna(subset=["npi", left_column])
        .rename(columns={left_column: left_value_col})
        .copy()
    )

    right = (
        right_df[["npi", right_column]]
        .dropna(subset=["npi", right_column])
        .rename(columns={right_column: right_value_col})
        .copy()
    )

    comparison = left.merge(
        right,
        on="npi",
        how="inner"
    )

    left_value = comparison[left_value_col].astype(str).str.strip()
    right_value = comparison[right_value_col].astype(str).str.strip()

    comparison["exact_match"] = left_value == right_value
    comparison["normalized_match"] = left_value.str.upper() == right_value.str.upper()

    return comparison

In [28]:
# Names
ehr_hr_first_name = compare_across_sources(
    "EHR", ehr_raw, "first_name",
    "HR", hr_raw, "first_name"
)

ehr_cred_first_name = compare_across_sources(
    "EHR", ehr_raw, "first_name",
    "CREDENTIALING", cred_raw, "legal_first_name"
)

display(ehr_hr_first_name[ehr_hr_first_name["exact_match"] == False].head(20))
display(ehr_cred_first_name[ehr_cred_first_name["exact_match"] == False].head(20))

,npi,first_name_ehr,first_name_hr,exact_match,normalized_match
6,1.316891e+09,Imani,IMANI,False,True
9,1.013457e+09,Gypsy,GYPSY,False,True
10,1.205886e+09,John,JOHN,False,True
12,1.265080e+09,Shu,SHU,False,True
13,1.538201e+09,Phillip,PHILLIP,False,True
22,1.871566e+09,Deborah,DEBORAH,False,True
26,1.619271e+09,Julia,JULIA,False,True
32,1.205592e+09,Adrian,ADRIAN,False,True
34,1.730557e+09,Mary,MARY,False,True
37,1.245044e+09,Angela,ANGELA,False,True


,npi,first_name_ehr,legal_first_name_credentialing,exact_match,normalized_match
4,1.649904e+09,Rachel,RACHEL,False,True
22,1.063009e+09,Kimberly,KIMBERLY,False,True
24,1.013457e+09,Gypsy,GYPSY,False,True
27,1.205886e+09,John,JOHN,False,True
31,1.457658e+09,Shelagh,SHELAGH,False,True
36,1.265080e+09,Shu,SHU,False,True
37,1.538201e+09,Phillip,PHILLIP,False,True
39,1.841928e+09,Rachelle,RACHELLE,False,True
51,1.205617e+09,Jorge,JORGE,False,True
52,1.790552e+09,Nancy,NANCY,False,True


In [29]:
# Credentials
ehr_hr_credential = compare_across_sources(
    "EHR", ehr_raw, "credential",
    "HR", hr_raw, "job_credential"
)

ehr_cred_credential = compare_across_sources(
    "EHR", ehr_raw, "credential",
    "CREDENTIALING", cred_raw, "credential"
)

display(ehr_hr_credential[ehr_hr_credential["exact_match"] == False].head(20))
display(ehr_cred_credential[ehr_cred_credential["exact_match"] == False].head(20))

,npi,credential_ehr,job_credential_hr,exact_match,normalized_match


,npi,credential_ehr,credential_credentialing,exact_match,normalized_match


In [30]:
# Specialty / taxonomy completeness across matched NPIs.
specialty_compare = (
    ehr_raw[["npi", "specialty_code"]]
    .dropna(subset=["npi"])
    .merge(
        hr_raw[["npi", "job_specialty_code"]].dropna(subset=["npi"]),
        on="npi",
        how="inner"
    )
    .merge(
        cred_raw[["npi", "taxonomy_code"]].dropna(subset=["npi"]),
        on="npi",
        how="inner"
    )
)

hr_specialty_gap = specialty_compare[
    specialty_compare["job_specialty_code"].isna() &
    (
        specialty_compare["specialty_code"].notna() |
        specialty_compare["taxonomy_code"].notna()
    )
]

display(hr_specialty_gap.head(20))

if not hr_specialty_gap.empty:
    add_finding(
        dataset="HR",
        column="job_specialty_code",
        issue="Specialty is missing for some providers where another source has specialty data",
        example=f"{len(hr_specialty_gap)} matched providers",
        planned_action="Retain source-level missingness and consider available specialty values during survivorship",
        stage="Data Quality"
    )

,npi,specialty_code,job_specialty_code,taxonomy_code
1,1.275485e+09,363LC0200X,NaN,363LC0200X
7,1.205886e+09,2085R0202X,NaN,2085R0202X
23,1.093473e+09,363LF0000X,NaN,363LF0000X
24,1.881440e+09,363LF0000X,NaN,363LF0000X
35,1.205678e+09,163WH1000X,NaN,163WH1000X
36,1.043771e+09,207N00000X,NaN,207N00000X
37,1.265098e+09,207R00000X,NaN,207R00000X
41,1.437518e+09,101Y00000X,NaN,101Y00000X
48,1.881678e+09,207VM0101X,NaN,207VM0101X
54,1.386766e+09,1223G0001X,NaN,1223G0001X


## 10. Review findings

In [31]:
# Review the final profiling log before moving into cleaning and standardization.
findings.sort_values(
    ["stage", "dataset", "column", "issue"]
).reset_index(drop=True)

,dataset,column,issue,example,planned_action,stage,status
0,Credentialing,npi,NPI inferred as numeric during import,1234567890.0,Convert to string and validate 10-digit format,Cleaning,Open
1,Credentialing,phone,Phone inferred as numeric during import,2.816416e+09,Convert to string and validate phone length,Cleaning,Open
2,Credentialing,zip,ZIP inferred as numeric during import,773386118,Convert to string before ZIP standardization,Cleaning,Open
3,EHR,npi,NPI inferred as numeric during import,1234567890.0,Convert to string and validate 10-digit format,Cleaning,Open
4,EHR,zip,ZIP inferred as numeric during import,773386118,Convert to string before ZIP standardization,Cleaning,Open
5,HR,npi,NPI inferred as numeric during import,1234567890.0,Convert to string and validate 10-digit format,Cleaning,Open
6,HR,job_specialty_code,Specialty code contains missing values,481 records (21.38% missing),Retain missing values and use specialty from o...,Data Quality,Open
7,HR,job_specialty_code,Specialty is missing for some providers where ...,166 matched providers,Retain source-level missingness and consider a...,Data Quality,Open
8,CREDENTIALING,npi,NPI contains missing values,51 records (2.0% missing),Retain missing values; do not impute NPI; use ...,Matching,Open
9,Credentialing,zip,ZIP codes are shared across multiple provider ...,782344504 appears on 13 records,Retain values and use ZIP as a supporting matc...,Matching,Open


## 11. Save the findings log

In [32]:
# Save the current profiling log. Re-running this cell replaces the existing file
# with the latest version of the findings DataFrame.

PROFILE_DIR = Path("../data/processed/profiling")
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

FINDINGS_PATH = PROFILE_DIR / "source_profiling_findings.csv"

findings_to_save = (
    findings
    .drop_duplicates(subset=["dataset", "column", "issue"])
    .sort_values(["dataset", "column", "issue"])
    .reset_index(drop=True)
)

findings_to_save.to_csv(
    FINDINGS_PATH,
    index=False
)

print(f"Saved {len(findings_to_save)} findings to:")
print(FINDINGS_PATH)

Saved 34 findings to:
../data/processed/profiling/source_profiling_findings.csv
